# Toxic Comment Classifier
### NLP Course Project

**Objective:** Build a binary text classifier to identify toxic online comments using classical NLP techniques and machine learning.

**Pipeline Overview:**
1. Load Dataset  
2. NLP Preprocessing (lowercase → clean → tokenize → stopword removal → lemmatize/stem)  
3. Feature Extraction (TF-IDF)  
4. Model Training (Logistic Regression)  
5. Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)  
6. Inference / Demo  


In [ ]:
import os, sys, re, pickle, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import nltk
# Point to bundled NLTK data
NLTK_DATA = os.path.join(os.getcwd(), "nltk_data")
if os.path.isdir(NLTK_DATA):
    nltk.data.path.insert(0, NLTK_DATA)
for pkg in ("punkt", "punkt_tab", "stopwords", "wordnet"):
    try: nltk.download(pkg, quiet=True)
    except: pass

from nltk.corpus import stopwords as sw_corpus
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              roc_auc_score, roc_curve)

print("All imports successful ✓")

## 1. Load Dataset

In [ ]:
df = pd.read_csv("data/train.csv")
print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df["toxic"].value_counts())
print(f"\nClass balance: {df['toxic'].mean()*100:.1f}% toxic")
df.head()

## 2. NLP Preprocessing

Classical NLP preprocessing involves several steps to normalize and clean raw text:

| Step | Purpose | Example |
|------|---------|---------|
| Lowercasing | Normalize case | `Hello` → `hello` |
| Punctuation removal | Remove noise | `you're!!` → `youre` |
| Tokenization | Split into words | `"hello world"` → `["hello", "world"]` |
| Stopword removal | Remove low-info words | removes `the`, `is`, `at` |
| Stemming | Root form (rule-based) | `running` → `run` |
| Lemmatization | Root form (linguistic) | `studies` → `study` |


In [ ]:
stemmer = PorterStemmer()

try:
    STOP_WORDS = set(sw_corpus.words("english"))
except Exception:
    STOP_WORDS = {"i","me","my","we","our","you","your","he","him","his","she","her",
                  "they","them","what","this","that","am","is","are","was","were","be",
                  "been","have","has","do","does","did","a","an","the","and","but","if",
                  "or","as","of","at","by","for","with","in","out","on","off","to","from"}

# Try lemmatizer, fallback to stemmer
try:
    lemmatizer = WordNetLemmatizer()
    lemmatizer.lemmatize("test")
    USE_LEMMA = True
    print("Using: WordNet Lemmatizer")
except:
    USE_LEMMA = False
    print("Using: Porter Stemmer (WordNet unavailable)")

def preprocess(text):
    # Step 1: Lowercase
    text = text.lower()
    # Step 2: Remove punctuation & special chars
    text = re.sub(r"[^a-z\s]", "", text)
    # Step 3: Tokenize
    tokens = word_tokenize(text)
    # Step 4: Remove stopwords + short tokens
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    # Step 5: Lemmatize or Stem
    if USE_LEMMA:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    else:
        tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

# Show preprocessing on sample
raw = "You're such an IDIOT!! Nobody likes your stupid comments, get lost!!!"
print(f"\nRaw:       {raw}")
print(f"Processed: {preprocess(raw)}")

raw2 = "I really enjoyed reading this thoughtful article. Thanks for sharing!"
print(f"\nRaw:       {raw2}")
print(f"Processed: {preprocess(raw2)}")

In [ ]:
print("Applying preprocessing pipeline to all comments...")
df["cleaned"] = df["comment_text"].apply(preprocess)
print(f"Done. {len(df)} comments preprocessed.\n")
df[["comment_text", "toxic", "cleaned"]].head(8)

## 3. Feature Extraction — TF-IDF

**TF-IDF** (Term Frequency – Inverse Document Frequency) converts text into numerical vectors.

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

- **TF** measures how frequently a term appears in a document  
- **IDF** penalizes terms that appear in many documents (common, low-information words)

We use `ngram_range=(1,2)` to capture both individual words and two-word phrases (bigrams),  
which helps the model learn phrases like *"shut up"* or *"go away"*.


In [ ]:
# Train/test split — 80% train, 20% test
# stratify=y ensures both splits preserve the class ratio
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df["cleaned"], df["toxic"],
    test_size=0.2, random_state=42, stratify=df["toxic"]
)
print(f"Train: {len(X_train_raw)} | Test: {len(X_test_raw)}")

# TF-IDF Vectorizer
vectorizer = TfidfVectorizer(
    max_features=10_000,   # top 10k most informative terms
    ngram_range=(1, 2),    # unigrams AND bigrams
    sublinear_tf=True,     # log normalization of term frequencies
    min_df=2,              # ignore very rare terms
)

X_train = vectorizer.fit_transform(X_train_raw)   # fit on train only
X_test  = vectorizer.transform(X_test_raw)        # apply same vocab to test

print(f"\nTF-IDF matrix shape (train): {X_train.shape}")
print(f"TF-IDF matrix shape (test) : {X_test.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

# Show top features
feature_names = vectorizer.get_feature_names_out()
print(f"\nSample features: {list(feature_names[:10])}")

## 4. Model Training — Logistic Regression

**Logistic Regression** predicts the probability of a comment being toxic using the sigmoid function:

$$P(y=1 | x) = \frac{1}{1 + e^{-w^T x}}$$

It is ideal for TF-IDF features because:
- Works well with high-dimensional sparse matrices  
- Outputs calibrated probabilities  
- Fast training and inference  
- Built-in L2 regularization prevents overfitting


In [ ]:
model = LogisticRegression(
    C=1.0,        # regularization strength (lower = more regularization)
    max_iter=1000,
    solver="lbfgs",
    random_state=42
)
model.fit(X_train, y_train)
print("Model trained successfully ✓")
print(f"Training samples: {X_train.shape[0]}")
print(f"Features: {X_train.shape[1]}")

## 5. Model Evaluation

### Metrics Explained

| Metric | Formula | Meaning |
|--------|---------|---------|
| **Accuracy** | (TP+TN)/(Total) | Overall correctness |
| **Precision** | TP/(TP+FP) | Quality of positive predictions |
| **Recall** | TP/(TP+FN) | Coverage of actual positives |
| **F1-Score** | 2·(P·R)/(P+R) | Harmonic mean of P and R |

In content moderation, **Recall** is critical — we want to catch as many toxic comments as possible.


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)

print("=" * 45)
print("      EVALUATION RESULTS")
print("=" * 45)
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  ROC-AUC   : {auc:.4f}")
print("=" * 45)
print()
print(classification_report(y_test, y_pred, target_names=["Non-Toxic", "Toxic"]))

## 6. Confusion Matrix

The confusion matrix shows the distribution of correct and incorrect predictions:

- **TP (True Positive)**: Toxic predicted as Toxic ✓  
- **TN (True Negative)**: Non-Toxic predicted as Non-Toxic ✓  
- **FP (False Positive)**: Non-Toxic predicted as Toxic ✗ (false alarm)  
- **FN (False Negative)**: Toxic predicted as Non-Toxic ✗ (missed toxicity)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
labels = ["Non-Toxic", "Toxic"]
im = axes[0].imshow(cm, cmap="Blues")
plt.colorbar(im, ax=axes[0])
axes[0].set(xticks=range(2), yticks=range(2), xticklabels=labels, yticklabels=labels,
            xlabel="Predicted Label", ylabel="True Label",
            title="Confusion Matrix")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i,j] > cm.max()/2 else "black",
                     fontsize=16, fontweight="bold")

# ── ROC Curve ─────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_val = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC = {auc_val:.3f}")
axes[1].plot([0,1],[0,1], "k--", lw=1, label="Random")
axes[1].fill_between(fpr, tpr, alpha=0.1, color="steelblue")
axes[1].set(xlabel="False Positive Rate", ylabel="True Positive Rate",
            title="ROC Curve", xlim=[0,1], ylim=[0,1.02])
axes[1].legend()

plt.tight_layout()
plt.savefig("models/evaluation_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plots saved to models/evaluation_plots.png")

## 7. Live Inference Demo

In [ ]:
def classify(text):
    cleaned  = preprocess(text)
    features = vectorizer.transform([cleaned])
    pred     = model.predict(features)[0]
    proba    = model.predict_proba(features)[0]
    label    = "Toxic" if pred == 1 else "Non-Toxic"
    conf     = proba[1] if pred == 1 else proba[0]
    icon     = "🔴" if pred == 1 else "🟢"
    return f"{icon} {label} (confidence: {conf*100:.1f}%)"

test_comments = [
    "You are such an idiot, nobody cares about your stupid opinion.",
    "I really enjoyed reading this article, very informative.",
    "Go kill yourself, worthless piece of garbage.",
    "This is a great community, everyone is so helpful.",
    "You filthy animal, you do not deserve to breathe.",
    "Could you please elaborate on that point a bit more?",
    "What a disgusting human being you are.",
    "I disagree with your approach, but I understand your reasoning.",
]

print("\n" + "=" * 65)
print("   SAMPLE PREDICTIONS")
print("=" * 65)
for comment in test_comments:
    result = classify(comment)
    print(f"\n  Comment: {comment[:60]}{'...' if len(comment)>60 else ''}")
    print(f"  Result : {result}")

## 8. Save Model with Pickle

In [ ]:
os.makedirs("models", exist_ok=True)

with open("models/toxic_model.pkl",      "wb") as f: pickle.dump(model, f)
with open("models/tfidf_vectorizer.pkl", "wb") as f: pickle.dump(vectorizer, f)

print("Model saved     → models/toxic_model.pkl")
print("Vectorizer saved → models/tfidf_vectorizer.pkl")
print("\nTo run the Streamlit app:")
print("   streamlit run app.py")